In [13]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacpassword1"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '100px'}), href='https://www.snhu.edu')),
    html.Center(html.P('Created by Phetmanee Thongkham')),
    html.Hr(),
    html.Div(
        
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}

    ),
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         page_size=10,
                         page_action='native',
                         sort_action='native',
                         sort_mode='multi',
                         filter_action='native',
                         row_selectable='single',
                         selected_rows=[0],
                         style_table={'overflowX': 'auto'},
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Output('datatable-id', 'filter_query'),
    Output('datatable-id', 'sort_by'),
    Output('datatable-id', 'page_current'),
    Input('filter-type', 'value'))
def update_dashboard(filter_type):
    # Query the existing CRUD module using the specification's inclusive ages.
    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': r'(^|/)(Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland)( Mix)?(/|$)', '$options': 'i'},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': r'(^|/)(German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler)( Mix)?(/|$)', '$options': 'i'},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': r'(^|/)(Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler)( Mix)?(/|$)', '$options': 'i'},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
    else:
        query = {}
    dff = pd.DataFrame.from_records(db.read(query))
    dff.drop(columns=['_id'], inplace=True, errors='ignore')
    data = dff.astype(object).where(pd.notnull(dff), None).to_dict('records')
    # Clear previous table filters and selections when changing a rescue profile.
    return data, ([0] if data else []), '', [], 0
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    dff = pd.DataFrame.from_records(df.to_dict('records') if viewData is None else viewData)
    if dff.empty or 'breed' not in dff:
        return [html.P('No animals match the selected filters.')]
    counts = dff['breed'].fillna('Unknown breed').value_counts().reset_index()
    counts.columns = ['breed', 'count']
    counts = counts.head(10)
    return [dcc.Graph(figure=px.pie(
        counts, names='breed', values='count', title='Top 10 Breeds'))]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        viewData = df.to_dict('records')
    if not viewData:
        return [html.P('No animals match the selected filters.')]

    dff = pd.DataFrame.from_records(viewData)
    row = index[0] if index and 0 <= index[0] < len(dff) else 0
    animal = dff.iloc[row]
    # Read coordinates by field name rather than depending on column order.
    try:
        latitude = float(animal.get('location_lat'))
        longitude = float(animal.get('location_long'))
        if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
            raise ValueError('Invalid coordinates')
    except (TypeError, ValueError):
        return [html.P('This animal has no valid location coordinates.')]
    name = animal.get('name')
    if pd.isna(name) or not name:
        name = 'Unnamed animal'
    breed = animal.get('breed')
    if pd.isna(breed) or not breed:
        breed = 'Unknown breed'

    return [
        dl.Map(style={'width': '100%', 'height': '500px'},
               center=[latitude, longitude], zoom=10, children=[
            dl.TileLayer(id='base-layer-id'),
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([
                    html.H1('Animal Name'),
                    html.P(str(name))
                ])
            ])
        ])
    ]
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://gururemote-paprikafalcon-3000.codio.io/proxy/8050/
